# Malaria-KI - komplette Pipeline (ein Notebook)

Dieses Notebook macht **alles von A bis Z**:

1. Daten einlesen aus `data/raw/` (Original-Datensatz)
2. Preprocessing: stratifizierter Split in **train / val / test** + Augmentierung
3. Training eines robusten ResNet50 (2-Phasen-Transfer-Learning)
4. Auswahl des besten Modells auf der **Validierung**
5. Ehrlicher Endtest auf dem **Test-Split** mit klarem Ergebnis

**Datensatz ablegen unter `data/raw/`** mit zwei Unterordnern. Erkannt werden:
`Parasitized` / `Uninfected` **oder** `infected` / `healthy`
(auch verschachtelt, z. B. `data/raw/cell_images/...`).

Einfach alle Zellen der Reihe nach ausfuehren (oben: *Run All*).


## 1. Setup & Konfiguration

In [ ]:
import os, json, time, random, copy
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from tqdm.auto import tqdm

# ---- Reproduzierbarkeit ----
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# ---- Pfade (funktioniert egal ob aus dem Projektordner oder aus notebooks/ gestartet) ----
BASE = Path.cwd()
if BASE.name == "notebooks":
    BASE = BASE.parent
RAW          = BASE / "data" / "raw"
CKPT_DIR     = BASE / "models" / "checkpoints"; CKPT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_DIR    = BASE / "models" / "final";       FINAL_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR    = BASE / "results" / "plots";       PLOTS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR  = BASE / "results" / "metrics";     METRICS_DIR.mkdir(parents=True, exist_ok=True)

# ---- Gateraet & Performance ----
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.set_float32_matmul_precision("high")

USE_AMP   = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16            # stabil & schnell auf der 5070 Ti
CH_LAST   = DEVICE == "cuda"

# ---- Bild / Daten ----
IMG_SIZE = 224
# ImageNet-Normalisierung: passt zum vortrainierten Modell -> robust.
MEAN = (0.485, 0.456, 0.406)
STD  = (0.229, 0.224, 0.225)

# ---- Training ----
BATCH_SIZE   = 128
NUM_WORKERS  = 8
HEAD_EPOCHS  = 3       # Phase A: nur Kopf
FINE_EPOCHS  = 25      # Phase B: gesamtes Netz (mit Early Stopping)
PATIENCE     = 6
LABEL_SMOOTH = 0.05
WEIGHT_DECAY = 1e-4
LR_HEAD      = 1e-3
LR_HEAD_FT   = 3e-4    # Kopf in Phase B
LR_BACKBONE  = 3e-5    # Backbone in Phase B (klein -> stabil)
MAX_GRAD_NORM = 1.0

CLASS_NAMES = ["healthy", "infected"]   # 0 = gesund, 1 = infiziert
print("Geraet:", DEVICE, "| AMP:", USE_AMP, "| Projektordner:", BASE)


## 2. Daten einlesen & in train / val / test aufteilen\n\nDer Split ist **stratifiziert** (gleiches Klassenverhaeltnis in jedem Teil) und mit festem Seed -> reproduzierbar. Es werden keine Dateien kopiert; gelesen wird direkt aus `data/raw/`.

In [ ]:
VALID_EXT = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}
HEALTHY_KEYS  = {"healthy", "uninfected", "normal"}
INFECTED_KEYS = {"infected", "parasitized", "parasite", "parasitised"}

def find_raw_root(root: Path) -> Path:
    """Steigt in einen einzelnen Zwischenordner ab (z.B. data/raw/cell_images/)."""
    if not root.exists():
        raise FileNotFoundError(f"Ordner fehlt: {root}  -> Datensatz nach data/raw/ legen")
    subs = [p for p in root.iterdir() if p.is_dir()]
    has_class = any(s.name.lower() in (HEALTHY_KEYS | INFECTED_KEYS) for s in subs)
    if not has_class and len(subs) == 1:
        return find_raw_root(subs[0])
    return root

def collect_samples():
    root = find_raw_root(RAW)
    subs = [p for p in root.iterdir() if p.is_dir()]
    healthy_dir  = next((p for p in subs if p.name.lower() in HEALTHY_KEYS), None)
    infected_dir = next((p for p in subs if p.name.lower() in INFECTED_KEYS), None)
    if healthy_dir is None or infected_dir is None:
        raise FileNotFoundError(
            f"Klassenordner nicht gefunden in {root}.\n"
            f"Erwartet: 'Uninfected'/'healthy' und 'Parasitized'/'infected'.\n"
            f"Gefunden: {[s.name for s in subs]}")
    samples = []
    for d, label in [(healthy_dir, 0), (infected_dir, 1)]:
        for p in sorted(d.iterdir()):
            if p.suffix.lower() in VALID_EXT:
                samples.append((p, label))
    print(f"Quelle: {root}")
    print(f"  healthy  ({healthy_dir.name}) : {sum(1 for _,l in samples if l==0):,}")
    print(f"  infected ({infected_dir.name}): {sum(1 for _,l in samples if l==1):,}")
    return samples

def stratified_split(samples, p_train=0.70, p_val=0.15):
    by = {0: [], 1: []}
    for s in samples:
        by[s[1]].append(s)
    rng = random.Random(SEED)
    train, val, test = [], [], []
    for label, items in by.items():
        rng.shuffle(items)
        n = len(items); n_tr = int(p_train*n); n_va = int(p_val*n)
        train += items[:n_tr]; val += items[n_tr:n_tr+n_va]; test += items[n_tr+n_va:]
    rng.shuffle(train); rng.shuffle(val); rng.shuffle(test)
    return train, val, test

samples = collect_samples()
train_s, val_s, test_s = stratified_split(samples)
print(f"\nSplit -> train: {len(train_s):,} | val: {len(val_s):,} | test: {len(test_s):,}")


## 3. Datasets, Augmentierung & DataLoader

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.5),          # Zellen haben kein Oben/Unten
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.15, 0.15, 0.1, 0.02),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class CellDataset(Dataset):
    def __init__(self, samples, tf):
        self.samples, self.tf = samples, tf
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        path, label = self.samples[i]
        return self.tf(Image.open(path).convert("RGB")), label

def make_loader(samples, tf, shuffle):
    return DataLoader(CellDataset(samples, tf), batch_size=BATCH_SIZE, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"),
                      persistent_workers=(NUM_WORKERS>0), drop_last=shuffle)

train_loader = make_loader(train_s, train_tf, True)
val_loader   = make_loader(val_s,   eval_tf,  False)
test_loader  = make_loader(test_s,  eval_tf,  False)
print("DataLoader bereit. Batches/Epoche (train):", len(train_loader))


## 4. Modell (ResNet50, vortrainiert)

In [ ]:
def build_model():
    m = models.resnet50(weights="DEFAULT")
    in_f = m.fc.in_features
    m.fc = nn.Sequential(
        nn.Linear(in_f, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(256, 2),
    )
    m = m.to(DEVICE)
    if CH_LAST:
        m = m.to(memory_format=torch.channels_last)
    return m

def set_backbone_trainable(model, trainable: bool):
    for name, p in model.named_parameters():
        if not name.startswith("fc."):
            p.requires_grad = trainable

def freeze_backbone_bn(model):
    """BatchNorm im Backbone auf eval -> ImageNet-Statistiken bleiben in Phase A stabil."""
    for name, mod in model.named_modules():
        if isinstance(mod, nn.BatchNorm2d) and not name.startswith("fc"):
            mod.eval()

model = build_model()
n_tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Modell bereit. Parameter gesamt: {sum(p.numel() for p in model.parameters()):,}")


## 5. Trainings- & Validierungsfunktionen\n\nWichtig: validiert wird das **echte Modell** im `eval()`-Modus (kein EMA-Versatz). Bestes Modell = kleinster Validierungs-Loss.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

def _prep(x):
    x = x.to(DEVICE, non_blocking=True)
    if CH_LAST:
        x = x.to(memory_format=torch.channels_last)
    return x

def train_one_epoch(model, loader, optimizer, freeze_bn=False, desc="train"):
    model.train()
    if freeze_bn:
        freeze_backbone_bn(model)
    loss_sum = correct = total = 0
    for x, y in tqdm(loader, desc=desc, leave=False):
        x, y = _prep(x), y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(x)
            loss = criterion(out, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()
        bs = y.size(0)
        loss_sum += loss.item()*bs
        correct  += (out.argmax(1) == y).sum().item()
        total    += bs
    return loss_sum/total, correct/total

@torch.no_grad()
def evaluate_loss(model, loader):
    model.eval()
    loss_sum = correct = total = 0
    for x, y in loader:
        x, y = _prep(x), y.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            out = model(x)
            loss = criterion(out, y)
        bs = y.size(0)
        loss_sum += loss.item()*bs
        correct  += (out.argmax(1) == y).sum().item()
        total    += bs
    return loss_sum/total, correct/total

@torch.no_grad()
def collect_probs(model, loader):
    """Gibt (labels, p_infected) zurueck."""
    model.eval()
    labels, probs = [], []
    for x, y in loader:
        x = _prep(x)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=USE_AMP):
            p = torch.softmax(model(x), dim=1)[:, 1]
        labels += y.tolist(); probs += p.float().cpu().tolist()
    return np.array(labels), np.array(probs)


## 6. Training\n\n**Phase A** trainiert nur den Kopf (Backbone eingefroren). **Phase B** taut das ganze Netz mit kleiner Lernrate (Cosine-Abfall) auf und feilt es fein. Bestes Modell wird laufend gespeichert; bei Stillstand greift Early Stopping.

In [ ]:
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

def log_epoch(tag, ep, n, tl, ta, vl, va, t0, best):
    print(f"{tag} {ep:2d}/{n} | train loss {tl:.4f} acc {ta:.3f} | "
          f"val loss {vl:.4f} acc {va:.3f} | {time.time()-t0:4.0f}s"
          + ("  <- bestes" if best else ""))

best_val = float("inf")
BEST_PATH = CKPT_DIR / "best.pth"

# ---------- Phase A: nur Kopf ----------
set_backbone_trainable(model, False)
opt_a = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                          lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
print("== Phase A: nur Kopf ==")
for ep in range(1, HEAD_EPOCHS+1):
    t0 = time.time()
    tl, ta = train_one_epoch(model, train_loader, opt_a, freeze_bn=True, desc=f"A {ep}")
    vl, va = evaluate_loss(model, val_loader)
    for k, v in zip(history, (tl, vl, ta, va)): history[k].append(v)
    improved = vl < best_val
    if improved:
        best_val = vl; torch.save(model.state_dict(), BEST_PATH)
    log_epoch("A", ep, HEAD_EPOCHS, tl, ta, vl, va, t0, improved)

# ---------- Phase B: gesamtes Netz feinjustieren ----------
set_backbone_trainable(model, True)
opt_b = torch.optim.AdamW([
    {"params": [p for n,p in model.named_parameters() if not n.startswith("fc.")], "lr": LR_BACKBONE},
    {"params": [p for n,p in model.named_parameters() if n.startswith("fc.")],     "lr": LR_HEAD_FT},
], weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt_b, T_max=FINE_EPOCHS)

print("\n== Phase B: Feinjustierung (gesamtes Netz) ==")
patience = 0
for ep in range(1, FINE_EPOCHS+1):
    t0 = time.time()
    tl, ta = train_one_epoch(model, train_loader, opt_b, freeze_bn=False, desc=f"B {ep}")
    vl, va = evaluate_loss(model, val_loader)
    sched.step()
    for k, v in zip(history, (tl, vl, ta, va)): history[k].append(v)
    improved = vl < best_val
    if improved:
        best_val = vl; torch.save(model.state_dict(), BEST_PATH); patience = 0
    else:
        patience += 1
    log_epoch("B", ep, FINE_EPOCHS, tl, ta, vl, va, t0, improved)
    if patience >= PATIENCE:
        print(f"Early Stopping (keine Verbesserung seit {PATIENCE} Epochen).")
        break

print(f"\nBestes Validierungs-Loss: {best_val:.4f}")


### Trainingsverlauf

In [ ]:
import matplotlib.pyplot as plt
ep = range(1, len(history["train_loss"])+1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
a1.plot(ep, history["train_loss"], label="Train"); a1.plot(ep, history["val_loss"], label="Val")
a1.set_title("Loss"); a1.set_xlabel("Epoche"); a1.legend(); a1.grid(alpha=.3)
a2.plot(ep, [a*100 for a in history["train_acc"]], label="Train")
a2.plot(ep, [a*100 for a in history["val_acc"]], label="Val")
a2.set_title("Accuracy (%)"); a2.set_xlabel("Epoche"); a2.legend(); a2.grid(alpha=.3)
plt.tight_layout(); plt.savefig(PLOTS_DIR/"training_history.png", dpi=150); plt.show()


## 7. Schwellenwert auf der Validierung bestimmen\n\nDer Entscheidungs-Schwellenwert wird **auf der Validierung** gewaehlt (nicht auf dem Test -> kein Schummeln). Ziel: hoher Recall (kein infizierter Fall soll uebersehen werden), dann bester F1.

In [ ]:
from sklearn.metrics import f1_score, recall_score

# bestes Modell laden
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))

val_labels, val_probs = collect_probs(model, val_loader)

def pick_threshold(labels, probs, min_recall=0.95):
    best_t, best_f1 = 0.5, -1.0
    for t in np.arange(0.05, 0.95, 0.01):
        preds = (probs >= t).astype(int)
        if recall_score(labels, preds, zero_division=0) < min_recall:
            continue
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, float(t)
    if best_f1 < 0:                      # falls min_recall nie erreicht
        for t in np.arange(0.05, 0.95, 0.01):
            f1 = f1_score(labels, (probs>=t).astype(int), zero_division=0)
            if f1 > best_f1: best_f1, best_t = f1, float(t)
    return round(best_t, 3)

THRESHOLD = pick_threshold(val_labels, val_probs)
print(f"Gewaehlter Schwellenwert (auf Val): {THRESHOLD}")


## 8. Endtest auf dem Test-Split -> Ergebnis\n\nDiese Daten hat das Modell nie gesehen. Das ist die ehrliche Endbewertung.

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             roc_curve, precision_recall_curve, classification_report)

test_labels, test_probs = collect_probs(model, test_loader)
test_preds = (test_probs >= THRESHOLD).astype(int)

tn, fp, fn, tp = confusion_matrix(test_labels, test_preds, labels=[0,1]).ravel()
acc  = accuracy_score(test_labels, test_preds)
rec  = recall_score(test_labels, test_preds, zero_division=0)      # Sensitivitaet
spec = tn / (tn + fp) if (tn+fp) else 0.0
prec = precision_score(test_labels, test_preds, zero_division=0)
f1   = f1_score(test_labels, test_preds, zero_division=0)
auc  = roc_auc_score(test_labels, test_probs)
ap   = average_precision_score(test_labels, test_probs)

# Bootstrap-Konfidenzintervall fuer Accuracy
rng = np.random.default_rng(SEED)
boot = [accuracy_score(test_labels[idx], test_preds[idx])
        for idx in (rng.integers(0, len(test_labels), len(test_labels)) for _ in range(1000))]
ci_low, ci_high = np.percentile(boot, 2.5), np.percentile(boot, 97.5)

print("\n" + "="*52)
print("            E R G E B N I S   ( T E S T )")
print("="*52)
print(f"  Testbilder        : {len(test_labels):,}")
print(f"  Schwellenwert     : {THRESHOLD}")
print("-"*52)
print(f"  Accuracy          : {acc*100:6.2f} %   (95% CI {ci_low*100:.2f}-{ci_high*100:.2f})")
print(f"  Recall / Sensitiv.: {rec*100:6.2f} %   (infizierte erkannt)")
print(f"  Specificity       : {spec*100:6.2f} %   (gesunde erkannt)")
print(f"  Precision         : {prec*100:6.2f} %")
print(f"  F1-Score          : {f1*100:6.2f} %")
print(f"  AUC               : {auc:6.4f}")
print(f"  Average Precision : {ap:6.4f}")
print("-"*52)
print(f"  Confusion Matrix:")
print(f"      richtig gesund (TN)   : {tn}")
print(f"      falsch infiziert (FP) : {fp}")
print(f"      uebersehen (FN)       : {fn}")
print(f"      richtig infiziert (TP): {tp}")
print("="*52)
print("\n" + classification_report(test_labels, test_preds, target_names=CLASS_NAMES, digits=4))


### Test-Plots: Confusion Matrix, ROC, Precision-Recall

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

cm = confusion_matrix(test_labels, test_preds, labels=[0,1])
im = axes[0].imshow(cm, cmap="Blues")
for (i,j), v in np.ndenumerate(cm):
    axes[0].text(j, i, str(v), ha="center", va="center",
                 color="white" if v > cm.max()/2 else "black", fontsize=13)
axes[0].set_xticks([0,1], CLASS_NAMES); axes[0].set_yticks([0,1], CLASS_NAMES)
axes[0].set_xlabel("Vorhergesagt"); axes[0].set_ylabel("Tatsaechlich"); axes[0].set_title("Confusion Matrix")

fpr, tpr, _ = roc_curve(test_labels, test_probs)
axes[1].plot(fpr, tpr, lw=2, label=f"AUC = {auc:.4f}")
axes[1].plot([0,1],[0,1],"--",color="gray")
axes[1].set_xlabel("False Positive Rate"); axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("ROC"); axes[1].legend(loc="lower right"); axes[1].grid(alpha=.3)

pr, rc, _ = precision_recall_curve(test_labels, test_probs)
axes[2].plot(rc, pr, lw=2, label=f"AP = {ap:.4f}")
axes[2].set_xlabel("Recall"); axes[2].set_ylabel("Precision")
axes[2].set_title("Precision-Recall"); axes[2].legend(loc="lower left"); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.savefig(PLOTS_DIR/"test_results.png", dpi=150); plt.show()


## 9. Finales Modell & Metriken speichern

In [ ]:
final_path = FINAL_DIR / "final_model.pth"
torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": "resnet50",
    "num_classes": 2,
    "class_names": CLASS_NAMES,
    "mean": list(MEAN), "std": list(STD),
    "image_size": IMG_SIZE,
    "threshold": THRESHOLD,
}, final_path)

metrics = {
    "n_test": int(len(test_labels)), "threshold": THRESHOLD,
    "accuracy": round(float(acc), 4), "accuracy_ci": [round(float(ci_low),4), round(float(ci_high),4)],
    "recall": round(float(rec), 4), "specificity": round(float(spec), 4),
    "precision": round(float(prec), 4), "f1": round(float(f1), 4),
    "auc": round(float(auc), 4), "ap": round(float(ap), 4),
    "confusion_matrix": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
}
(METRICS_DIR / "test_metrics.json").write_text(json.dumps(metrics, indent=2))

print("Gespeichert:")
print(" -", final_path)
print(" -", METRICS_DIR / "test_metrics.json")
print(" - Plots in", PLOTS_DIR)
print(f"\nFazit: Test-Accuracy {acc*100:.2f} %, Recall {rec*100:.2f} %, AUC {auc:.4f}")
